In [ ]:
print("test")
import sys
import json
import math
import sqlite3
from datetime import datetime
from kafka import KafkaConsumer

with open("config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

BOOTSTRAP_SERVER = config["kafka"]["bootstrap_server"]
TOKENS_TOPIC = config["kafka"]["tokens_topic"]
BASELINES_DB = config["baselines_db_path"]

#load baselines into memory for fast lookups
print("attempritng to connect to db")
conn = sqlite3.connect(BASELINES_DB)
cursor = conn.cursor()
cursor.execute("SELECT * FROM token_baselines")
#currently loads all token baselines for all hours. can be configured
baseline_lookup = {row[0]: list(row[1:]) for row in cursor.fetchall()}
conn.close()

print(f"Loaded baselines for {len(baseline_lookup)} tokens.")

In [ ]:
#this function evaluates a token for 2 diffrent levels
#trend: a rise in apperenses, but over time. this signals a rise in popularity for example
#spike: a mommentery rise, very fast, this signals an event.

#all calculations scale with baseline, and have a floor for rare words
#the function returns a packet of informations that can be sent to diffrent notifications functions
bucket_state = {}

def evaluate_token(token, ts, baseline_hourly):
    #add to token list if needed
    if token not in bucket_state:
        bucket_state[token] = {
            "spike_level": 0.0,
            "trend_level": 0.0,
            "is_spike": False,
            "is_trend": False,
            "last_ts": ts
        }

    state = bucket_state[token]
    delta_t = max(0.0, ts - state["last_ts"])

    #calculate drain for the 2 levels. spike drains fast and trend slower
    #added a floor to drain rate, to handle rare words
    drain_spike = max(6.0 / 3600.0, 4.0 * (baseline_hourly / 3600.0)) * delta_t
    drain_trend = max(0.5 / 3600.0, 1.2 * (baseline_hourly / 3600.0)) * delta_t

    #calculate the new levels
    spike_level = max(0.0, state["spike_level"] - drain_spike + 1.0)
    trend_level = max(0.0, state["trend_level"] - drain_trend + 1.0)

    #thresh holds have floors to avoid rare words getting to many spikes
    #spike thresh hold is lower than trend
    baseline_sqrt = math.sqrt(baseline_hourly)
    thresh_spike = max(7.0, 3.0 * baseline_sqrt)
    thresh_trend = max(15.0, 5.0 * baseline_sqrt)

    spike_alert = False
    trend_alert = False

    #update boolien values
    if not state["is_spike"] and spike_level >= thresh_spike:
        state["is_spike"] = True
        spike_alert = True
    elif state["is_spike"] and spike_level < (thresh_spike * 0.75):
        ##0.75 for a buffer, we dont want to easly switch between modes
        state["is_spike"] = False

    if not state["is_trend"] and trend_level >= thresh_trend:
        state["is_trend"] = True
        trend_alert = True
    elif state["is_trend"] and trend_level < (thresh_trend * 0.75):
        state["is_trend"] = False

    #update state value to match
    state["spike_level"] = spike_level
    state["trend_level"] = trend_level
    state["last_ts"] = ts

    return {
        "token": token,
        "ts": ts,
        "spike_alert": spike_alert,
        "trend_alert": trend_alert,
        "spike_level": spike_level,
        "trend_level": trend_level,
        "thresh_spike": thresh_spike,
        "thresh_trend": thresh_trend,
        "is_spike": state["is_spike"],
        "is_trend": state["is_trend"],
        "baseline_hourly": baseline_hourly
    }

In [ ]:
#recives results from evaluate token and prints to screen. can be changed to any channle like telgram, notifications and so on
def notify_user(eval_res):
    time_str = datetime.fromtimestamp(eval_res["ts"]).strftime("%Y-%m-%d %H:%M:%S")

    if eval_res["spike_alert"]:
        print(f"[{time_str}] NOTIFY SPIKE: '{eval_res['token']}'")

    if eval_res["trend_alert"]:
        print(f"[{time_str}] NOTIFY TREND: '{eval_res['token']}'")

In [ ]:
debug_stats = {
    "total_tokens": 0,
    "spike_alerts": 0,
    "trend_alerts": 0
}
#a more rubust mode for developers to see more info on run time
def notify_user_debug(eval_res):
    debug_stats["total_tokens"] += 1
    time_str = datetime.fromtimestamp(eval_res["ts"]).strftime("%Y-%m-%d %H:%M:%S")

    if eval_res["spike_alert"]:
        debug_stats["spike_alerts"] += 1
        print(f"[{time_str}] NOTIFY SPIKE: Token: '{eval_res['token']}' | Level: {eval_res['spike_level']:.2f} / {eval_res['thresh_spike']:.1f} | Base: {eval_res['baseline_hourly']:.3f}/h")

    if eval_res["trend_alert"]:
        debug_stats["trend_alerts"] += 1
        print(f"[{time_str}] NOTIFY TREND: Token: '{eval_res['token']}' | Level: {eval_res['trend_level']:.2f} / {eval_res['thresh_trend']:.1f} | Base: {eval_res['baseline_hourly']:.3f}/h")

    #summery of the last 1000 tokens
    if debug_stats["total_tokens"] % 1000 == 0:
        active_spikes = [k for k, v in bucket_state.items() if v["is_spike"]]
        active_trends = [k for k, v in bucket_state.items() if v["is_trend"]]
        print(
            f"[{time_str}] [SUMMARY] Counted: {debug_stats['total_tokens']:,} tokens | "
            f"Notified: {debug_stats['spike_alerts']} spikes, {debug_stats['trend_alerts']} trends | "
            f"Active: {len(active_spikes)} spike(s), {len(active_trends)} trend(s)"
        )

In [ ]:
consumer = KafkaConsumer(
    TOKENS_TOPIC,
    bootstrap_servers=BOOTSTRAP_SERVER,
    group_id="leaky-bucket-evaluator-group",
    auto_offset_reset='latest',
    enable_auto_commit=True,
    key_deserializer=lambda k: k.decode('utf-8') if k else None,
    value_deserializer=lambda v: json.loads(v.decode('utf-8'))
)

print("Stage 2 worker live")

try:
    for msg in consumer:
        token = msg.key
        if not token:
            continue

        ts = msg.value.get("ts", int(datetime.now().timestamp()))
        hour = datetime.fromtimestamp(ts).hour
        baseline_hourly = baseline_lookup.get(token, [0.01] * 24)[hour]

        notify_user_debug(evaluate_token(token, ts, baseline_hourly))
        #notify_user(evaluate_token(token, ts, baseline_hourly))


except KeyboardInterrupt:
    print("\nWorker stopped.")
finally:
    consumer.close()